<!-- SPDX-License-Identifier: GPL-2.0-only -->
# 55 Protocol

The notebook judges the wire between the page and its Web Worker without a browser. It reads the
script of the worker `web/js/worker.js`, the methods of the module
`crates/bausteine-web/src/worker.rs`, the transport of the page `web/js/bridge.js` and the reference
`Documentation/worker.md` as text, and checks that every message has a method and every method a
caller. It sends every GraphQL request the client of the page `web/js/store.js` writes to the engine
`bausteine-graphql`, which boots the service as the worker does, so a query of the page that the
schema refuses fails here. Last it checks that both sides speak one version of the protocol. A cell
that proves an entry of the register `Documentation/quirks.md` carries the tag its column Test
names, and its checks name the ID of the entry.

In [ ]:
engine_binary = None  # the path of bausteine-graphql, the debug build by default

In [ ]:
import sys, json, math
sys.path.insert(0, '..')
from harness import *

In [ ]:
import re
WEB = ROOT / 'web' / 'js'
worker_js = (WEB / 'worker.js').read_text(encoding='utf-8')
bridge_js = (WEB / 'bridge.js').read_text(encoding='utf-8')
store_js = (WEB / 'store.js').read_text(encoding='utf-8')
worker_rs = (ROOT / 'crates' / 'bausteine-web' / 'src' / 'worker.rs').read_text(encoding='utf-8')
reference = (ROOT / 'Documentation' / 'worker.md').read_text(encoding='utf-8')

def braced(text, at):
    # The text from the brace at or after `at` to the brace that closes it.
    start = text.index('{', at)
    depth = 0
    for i in range(start, len(text)):
        depth += {'{': 1, '}': -1}.get(text[i], 0)
        if depth == 0:
            return text[start:i + 1]
    raise Failure('a brace is not closed')

def function(text, name):
    # The body of the function `name` of a script.
    return braced(text, re.search(r'function ' + name + r'\(', text).end())

def table(markdown, heading):
    # The first column of the first table under the heading that starts with `heading`.
    section = markdown[markdown.index(heading):]
    section = section[:section.index('\n## ', 1)] if '\n## ' in section[1:] else section
    rows = [line for line in section.splitlines() if line.startswith('| `')]
    return [re.match(r'\| `([^`]+)`', row).group(1) for row in rows]

## 1. Every message has a method and every method a caller

The worker answers nine messages of the page, each in one `case` of its dispatch: `init` starts the
module and opens WebGPU with `Worker.create`, `state` calls one setter per field, and every other
message calls the method of its name. The public methods of `#[wasm_bindgen] impl Worker` are those
the protocol names, plus `revision`, which each frame reads, and the script calls every one of them.
The bridge sends exactly the nine messages, the worker posts exactly the messages the bridge reads,
and the tables of `Documentation/worker.md` name the same messages and the fields of `state` in the
order the worker applies them.

In [ ]:
block = braced(worker_rs, re.search(r'#\[wasm_bindgen\]\s*impl Worker\b', worker_rs).end())
methods = re.findall(r'^\s*pub (?:async )?fn (\w+)', block, re.M)
cases = re.findall(r"case '(\w+)'", function(worker_js, 'handle'))
called = set(re.findall(r'\bengine\.(\w+)\(', worker_js)) | set(re.findall(r'\bEngine\.(\w+)\(', worker_js))
check(cases == ['init', 'graphql', 'state', 'pick', 'bounds', 'snapshot', 'render', 'dump', 'close'], 'the worker answers the nine messages of the protocol', cases)
check(sorted(methods) == sorted(['create', 'schema', 'graphql', 'show', 'set_selected', 'set_preview', 'resize', 'set_viewport',
      'set_ortho', 'set_orbit', 'set_dark', 'set_classic', 'set_oblique_zoom', 'set_oblique_pan', 'set_turn', 'set_inverted', 'set_raytrace',
      'render', 'revision', 'pick', 'bounds', 'snapshot', 'dump', 'close']), 'the module offers the methods of the protocol', methods)
check([c for c in cases if c not in ('init', 'state') and c not in methods] == [], 'every message but init and state names a method of the module')
check('create' in called and 'create' in methods, 'init starts the module with Worker.create')
setters = set(re.findall(r'\bengine\.(\w+)\(', function(worker_js, 'applyState')))
check(setters == {m for m in methods if m.startswith('set_')} | {'resize', 'show'}, 'state calls every setter of the module, resize and show', sorted(setters))
check(sorted(set(methods) - called) == [] and sorted(called - set(methods)) == [], 'the script calls every public method of the module and no other',
      sorted(set(methods) - called), sorted(called - set(methods)))
sent = set(re.findall(r"\bcall\('(\w+)'", bridge_js)) | set(re.findall(r"postMessage\(\{ type: '(\w+)'", bridge_js))
check(sent == set(cases), 'the bridge sends exactly the nine messages', sorted(sent))
posted = set(re.findall(r"\btype: '(\w+)'", worker_js))
read = set(re.findall(r"m\.type === '(\w+)'", bridge_js))
check(posted == {'ready', 'graphql', 'frame', 'pick', 'bounds', 'snapshot', 'dump', 'error'}, 'the worker posts the eight messages of the protocol', sorted(posted))
check(read <= posted, 'every message the bridge tells apart is one the worker posts', sorted(read))
check(table(reference, '## 1.') == cases, 'Documentation/worker.md names the messages to the worker in the order of the dispatch', table(reference, '## 1.'))
check(sorted(table(reference, '## 3.')) == sorted(posted), 'Documentation/worker.md names the messages to the page', table(reference, '## 3.'))
fields = list(dict.fromkeys(re.findall(r'\bm\.(\w+)', function(worker_js, 'applyState'))))
check(table(reference, '## 2.') == fields, 'Documentation/worker.md names the fields of state in the order the worker applies them', fields)

## 2. Every query of the page answers

The client of the page writes its GraphQL requests in `store.js`, partly from the constants of the
fields of an item and of a scene. The notebook reads every request that `WorkerStore` sends, fills
in those constants, gives each the variables the store gives it for a scene made from the example
`haus`, and sends it to the engine in the order a visit of the page would: create, read, change,
replace, remove, rename, delete. Every answer comes without `errors`. The bridge sends the body
`{query, variables}` as JSON, the body `Engine` of the harness sends as well. The client of a
GraphQL server over HTTP that `store.js` held is gone: the page posts every request to the Web
Worker, and the texts of the queries stay those a server could answer later.

In [ ]:
constants = dict(re.findall(r"^const (\w+) = [`']([^`']*)[`'];", store_js, re.M | re.S))
def resolved(text):
    for _ in range(5):
        text = re.sub(r'\$\{(\w+)\}', lambda m: constants[m.group(1)], text)
    return ' '.join(text.split())
requests = []
for m in re.finditer(r"this\.request\(\s*[`']([^`']*)[`']", store_js, re.S):
    method = re.findall(r'async (\w+)\(', store_js[:m.start()])[-1]
    requests.append((method, resolved(m.group(1))))
print(len(requests), 'requests in store.js:', ', '.join(m for m, _ in requests))
check(len(requests) >= 12 and all('${' not in q for _, q in requests), 'every request of store.js resolves to a text of GraphQL', requests)
queries = [q for _, q in requests]

def root(query):
    return re.match(r'\s*(?:query|mutation)?\s*(?:\([^)]*\))?\s*\{\s*(\w+)', query).group(1)
def declared(query):
    head = re.match(r'\s*(?:query|mutation)\s*\(([^)]*)\)', query)
    return re.findall(r'\$(\w+)\s*:', head.group(1)) if head else []

ORDER = ['createScene', 'textures', 'scene', 'addItem', 'updateItem', 'moveItem', 'replaceItems', 'removeItem', 'renameScene', 'deleteScene']
unknown = sorted({root(q) for q in queries} - set(ORDER))
check(unknown == [], 'the notebook knows the variables of every root field store.js asks for', unknown)

def as_input(item, keep_id):
    # The input of an item as toInput of store.js writes it.
    out = {k: item[k] for k in ('id', 'name', 'kind', 'parent', 'position', 'size', 'rotation', 'color', 'texture', 'innerRadius', 'source')}
    if not keep_id:
        del out['id']
    return out

service = Engine(engine_binary)
state = {}
def variables(query):
    field = root(query)
    if field == 'createScene':
        pool = {'name': 'Probe', 'example': 'haus'}
    elif field == 'textures':
        pool = {}
    elif field == 'scene':
        pool = {'id': state['scene'], 'eye': {'x': 10, 'y': 10, 'z': 5}, 'target': None, 'classic': True, 'date': '06.03.2002', 'time': '21:03:31'}
    elif field == 'addItem':
        pool = {'scene': state['scene'], 'input': as_input(state['items'][0], False) | {'name': 'Probe'}, 'after': None}
    elif field == 'updateItem':
        pool = {'scene': state['scene'], 'id': state['added'], 'input': {'name': 'Probe 2'}}
    elif field == 'moveItem':
        pool = {'scene': state['scene'], 'id': state['added'], 'after': None}
    elif field == 'replaceItems':
        pool = {'scene': state['scene'], 'items': [as_input(i, True) for i in state['items']]}
    elif field == 'removeItem':
        pool = {'scene': state['scene'], 'id': state['added']}
    else:
        pool = {'id': state['scene'], 'name': 'Probe 3'}
    names = declared(query)
    missing = [n for n in names if n not in pool]
    if missing:
        raise Failure(f'the notebook gives no value to the variables {missing} of {field}')
    return {n: pool[n] for n in names}

for method, query in sorted(requests, key=lambda r: ORDER.index(root(r[1]))):
    body = service.graphql(query, variables(query))
    field = root(query)
    check('errors' not in body and body.get('data', {}).get(field) is not None, f'R-50 WorkerStore.{method} asks {field} without errors',
          query, json.dumps(body, ensure_ascii=False)[:1500])
    data = body['data'][field]
    if field == 'createScene':
        state['scene'], state['items'] = data['id'], data['items']
    elif field == 'addItem':
        state['added'] = data['id']
    elif isinstance(data, dict) and 'items' in data:
        state['items'] = data['items']
service.stop()
check(re.search(r'JSON\.stringify\(\{\s*query,\s*variables\s*\}\)', bridge_js) is not None, 'the bridge sends the body {query, variables} that Engine sends')
classes = re.findall(r'^export class (\w+)', store_js, re.M)
check('fetch(' not in store_js and 'RemoteStore' not in store_js and classes == ['WorkerStore', 'Shelf'],
      'R-50 store.js holds no client of a server over HTTP, only WorkerStore and the Shelf of localStorage', classes)
check(re.search(r'async request\([^)]*\)\s*\{\s*const body = await this\.bridge\.graphql\(', store_js) is not None
      and re.search(r"async graphql\([^)]*\)\s*\{[^}]*this\.call\('graphql'", bridge_js) is not None,
      'R-50 WorkerStore posts every request through the bridge to the Web Worker')

## 3. One version of the protocol

The bridge and the worker each hold the version of the protocol, the worker answers it in `ready`,
and the bridge compares it with its own and calls its failure callback with a German message where
they differ, so a page and a worker of different builds never talk past each other.

In [ ]:
page = re.search(r'^export const PROTOCOL = (\d+);$', bridge_js, re.M)
worker = re.search(r'^const PROTOCOL = (\d+);$', worker_js, re.M)
check(page is not None and worker is not None, 'bridge.js and worker.js each hold the constant PROTOCOL')
check(page.group(1) == worker.group(1) == '1', 'both speak the protocol 1', page.group(1), worker.group(1))
check(re.search(r"type: 'ready'[^}]*protocol: PROTOCOL", worker_js) is not None, 'the worker answers its version in ready')
check('this.protocol !== PROTOCOL' in bridge_js, 'the bridge compares the version of ready with its own')